THIS NOTEBOOK IS FOR ALL GLM REPLATED MODELS
Block 1: GLM limited at 3 predictors
Block 2: Simplified GLM for reduced computational load
Block 3: Full scale GLM to be used if unlimited computational power is available

Block 1: GLM-3

In [ ]:
"""
Script to run a RAM-efficient GLM model on yearly Parquet files (1957–2016),
replicating Kelly, Gu, and Xiu (2020) using all predictors (~600).
Reduces computational load with low maxiter and single train-test split.
Avoids recursion issues with matrix-based GLM. Optimized for 8 cores, 64GB RAM.
"""
import os
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
import numpy as np
import gc
import logging

# Config
DATA_DIR = "INSERT DATA PATH HERE"
RESULTS_DIR = "INSERT OUTPUT PATH HERE"
RESULTS_FILE = os.path.join(RESULTS_DIR, "year_results.csv")
METRICS_FILE = os.path.join(RESULTS_DIR, "metrics.csv")
START_YEAR = 1957
END_YEAR = 2016
TARGET = 'ret_excess'
TEST_SIZE = 0.2
TOP_N_PREDICTORS = 5
RISK_FREE_RATE = 0.0
PERIODS_PER_YEAR = 12  # Monthly data
MAX_ITER = 20  # Reduced for efficiency
VARIANCE_THRESHOLD = 1e-5  # Remove very low-variance predictors

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def get_predictors(parquet_file):
    """Dynamically identify predictor columns, excluding non-predictors."""
    df = pd.read_parquet(parquet_file, engine='pyarrow')
    exclude_cols = [
        'permno', 'month', 'mktcap_lag', 'ret_excess',
        '__index_level_0__', '__fragment_index', '__batch_index',
        '__last_in_fragment', '__filename'
    ]
    predictors = [col for col in df.columns if col not in exclude_cols]
    del df
    gc.collect()
    return predictors

def setup_results_files():
    """Create or clear results and metrics files with headers."""
    if not os.path.exists(RESULTS_DIR):
        os.makedirs(RESULTS_DIR)
    
    if not os.path.exists(RESULTS_FILE):
        pd.DataFrame(columns=['year', 'r2', 'sharpe_ratio']).to_csv(RESULTS_FILE, index=False)
    
    if not os.path.exists(METRICS_FILE):
        pd.DataFrame(columns=[
            'year', 'mse', 'r2', 'tss', 'ess', 'sharpe_ratio', 'portfolio_return',
            'hit_ratio', 'n_observations', 'n_predictors'
        ]).to_csv(METRICS_FILE, index=False)
    
    logger.info(f"Results will be saved to {RESULTS_FILE}")
    logger.info(f"Metrics will be saved to {METRICS_FILE}")

def compute_portfolio_metrics(y_true, y_pred, year):
    """Compute long-short portfolio returns, Sharpe ratio, and hit ratio."""
    df = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred})
    df = df.sort_values('y_pred', ascending=False)
    top_quintile = df.iloc[:int(len(df) * 0.2)]
    bottom_quintile = df.iloc[-int(len(df) * 0.2):]
    
    portfolio_return = top_quintile['y_true'].mean() - bottom_quintile['y_true'].mean()
    annualized_return = portfolio_return * PERIODS_PER_YEAR
    portfolio_std = (top_quintile['y_true'] - bottom_quintile['y_true']).std() * np.sqrt(PERIODS_PER_YEAR)
    sharpe_ratio = (annualized_return - RISK_FREE_RATE) / portfolio_std if portfolio_std > 0 else np.nan
    hit_ratio = (top_quintile['y_true'] > bottom_quintile['y_true']).mean()
    
    return portfolio_return, sharpe_ratio, hit_ratio

def fit_glm(train_df, predictors):
    """Fit matrix-based GLM with reduced maxiter."""
    try:
        X_train = train_df[predictors]
        # Remove low-variance predictors
        selector = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
        X_train = selector.fit_transform(X_train)
        selected_predictors = [predictors[i] for i in selector.get_support(indices=True)]
        
        X_train = sm.add_constant(X_train)  # Add intercept
        y_train = train_df[TARGET]
        model = sm.GLM(y_train, X_train, family=sm.families.Gaussian()).fit(maxiter=MAX_ITER, cov_type='nonrobust')
        return model, selected_predictors
    except Exception as e:
        logger.error(f"GLM fitting failed: {str(e)}")
        return None, None

def process_year(year, predictors):
    """Process a single year's Parquet file and compute metrics."""
    parquet_file = os.path.join(DATA_DIR, f"year_{year}.parquet")
    
    if not os.path.exists(parquet_file):
        logger.warning(f"Parquet file for year {year} not found")
        return None
    
    df = None
    train_df = None
    test_df = None
    
    try:
        columns = [TARGET] + predictors
        df = pd.read_parquet(parquet_file, columns=columns, engine='pyarrow')
        logger.info(f"Loaded {len(df)} rows for year {year} with {len(predictors)} predictors")
        
        if df[columns].isnull().any().any():
            logger.warning(f"Missing values in year {year}, imputing with mean")
            df.fillna(df.mean(), inplace=True)
        
        train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=42)
        
        # Fit GLM
        model, selected_predictors = fit_glm(train_df, predictors)
        if model is None or selected_predictors is None:
            return None
        
        # Predict
        X_test = test_df[selected_predictors]
        selector = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
        X_test = selector.fit_transform(X_test)  # Apply same thresholding
        X_test = sm.add_constant(X_test)
        y_pred = model.predict(X_test)
        y_true = test_df[TARGET]
        
        r2 = r2_score(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        y_mean = y_true.mean()
        tss = ((y_true - y_mean) ** 2).sum()
        ess = ((y_pred - y_mean) ** 2).sum()
        
        portfolio_return, sharpe_ratio, hit_ratio = compute_portfolio_metrics(y_true, y_pred, year)
        
        p_values = model.pvalues[1:]  # Exclude intercept
        top_indices = np.argsort(p_values)[:TOP_N_PREDICTORS]
        top_predictors = [selected_predictors[i] for i in top_indices]
        top_p_values = p_values[top_indices]
        
        predictions_df = pd.DataFrame({
            'actual': y_true,
            'predicted': y_pred
        })
        predictions_file = os.path.join(RESULTS_DIR, f"predictions_{year}.csv")
        predictions_df.to_csv(predictions_file, index=False)
        
        predictors_df = pd.DataFrame({
            'predictor': top_predictors,
            'p_value': top_p_values
        })
        predictors_file = os.path.join(RESULTS_DIR, f"top_predictors_{year}.csv")
        predictors_df.to_csv(predictors_file, index=False)
        
        portfolio_df = pd.DataFrame({
            'year': [year],
            'portfolio_return': [portfolio_return]
        })
        portfolio_file = os.path.join(RESULTS_DIR, f"portfolio_{year}.csv")
        portfolio_df.to_csv(portfolio_file, index=False)
        
        logger.info(f"Year {year}: Out-of-sample R² = {r2:.6f}, Sharpe = {sharpe_ratio:.6f}")
        
        return {
            'year': year,
            'r2': r2,
            'sharpe_ratio': sharpe_ratio,
            'mse': mse,
            'tss': tss,
            'ess': ess,
            'portfolio_return': portfolio_return,
            'hit_ratio': hit_ratio,
            'n_observations': len(df),
            'n_predictors': len(selected_predictors),
            'top_predictors': top_predictors
        }
    
    except Exception as e:
        logger.error(f"Error processing year {year}: {str(e)}")
        return None
    finally:
        if df is not None:
            del df
        if train_df is not None:
            del train_df
        if test_df is not None:
            del test_df
        gc.collect()

def save_results(result):
    """Append results to CSV files."""
    if result is None:
        return
    
    result_df = pd.DataFrame({
        'year': [result['year']],
        'r2': [result['r2']],
        'sharpe_ratio': [result['sharpe_ratio']]
    })
    result_df.to_csv(RESULTS_FILE, mode='a', header=False, index=False)
    
    metrics_df = pd.DataFrame({
        'year': [result['year']],
        'mse': [result['mse']],
        'r2': [result['r2']],
        'tss': [result['tss']],
        'ess': [result['ess']],
        'sharpe_ratio': [result['sharpe_ratio']],
        'portfolio_return': [result['portfolio_return']],
        'hit_ratio': [result['hit_ratio']],
        'n_observations': [result['n_observations']],
        'n_predictors': [result['n_predictors']]
    })
    metrics_df.to_csv(METRICS_FILE, mode='a', header=False, index=False)

def main():
    logger.info("Starting GLM processing for yearly Parquet files (1957–2016)")
    
    setup_results_files()
    
    # Get predictors from the first available Parquet file
    first_parquet = os.path.join(DATA_DIR, f"year_{START_YEAR}.parquet")
    if not os.path.exists(first_parquet):
        logger.error(f"First Parquet file {first_parquet} not found")
        return
    predictors = get_predictors(first_parquet)
    logger.info(f"Using {len(predictors)} predictors")
    
    # Process years sequentially
    for year in range(START_YEAR, END_YEAR + 1):
        logger.info(f"Processing year {year}")
        result = process_year(year, predictors)
        save_results(result)
    
    logger.info("Processing complete")
    
    try:
        results_df = pd.read_csv(RESULTS_FILE)
        metrics_df = pd.read_csv(METRICS_FILE)
        
        logger.info("\nFinal Out-of-Sample R² Statistics:")
        logger.info(f"Count: {len(results_df)}")
        logger.info(f"Min R²: {results_df['r2'].min():.6f}")
        logger.info(f"Max R²: {results_df['r2'].max():.6f}")
        logger.info(f"Mean R²: {results_df['r2'].mean():.6f}")
        
        logger.info("\nFinal Sharpe Ratio Statistics:")
        logger.info(f"Min Sharpe: {results_df['sharpe_ratio'].min():.6f}")
        logger.info(f"Max Sharpe: {results_df['sharpe_ratio'].max():.6f}")
        logger.info(f"Mean Sharpe: {results_df['sharpe_ratio'].mean():.6f}")
        
        logger.info("\nFinal Metrics Summary:")
        logger.info(metrics_df.describe().to_string())
    except Exception as e:
        logger.error(f"Error reading results: {e}")

if __name__ == "__main__":
    main()

Block 2: Simplifed GLM

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings("ignore")

# PARAMETERS
DATA_DIR = 'INSERT DATA PATH HERE'
RESULTS_DIR = 'INSERT OUTPUT PATH HERE'
VALIDATION_LENGTH = 12
START_YEAR = 1957
END_YEAR = 2016

os.makedirs(RESULTS_DIR, exist_ok=True)

# Load Data
full_data = pd.concat([pd.read_parquet(f).assign(year=int(os.path.basename(f).split('_')[1].split('.')[0]))
                       for f in glob.glob(os.path.join(DATA_DIR, 'year_*.parquet'))], ignore_index=True)
print(f"Loaded data shape: {full_data.shape}")

# Features and Target
TARGET = 'ret_excess'
DROP_COLS = ['ret_excess', 'year', 'month', 'permno']
FEATURES = [col for col in full_data.columns if col not in DROP_COLS]
full_data = full_data.dropna(subset=[TARGET])

# Remove zero-variance columns
zero_var_cols = full_data[FEATURES].nunique()[full_data[FEATURES].nunique() <= 1].index.tolist()
FEATURES = [col for col in FEATURES if col not in zero_var_cols]

# Estimation Periods
def create_estimation_periods(start_year, end_year, validation_length):
    return pd.DataFrame([{
        'oos_year': y,
        'training_start': start_year,
        'training_end': y - validation_length - 1,
        'validation_start': y - validation_length,
        'validation_end': y - 1
    } for y in range(1987, end_year + 1)])

estimation_periods = create_estimation_periods(START_YEAR, END_YEAR, VALIDATION_LENGTH)

for _, period in estimation_periods.iterrows():
    print(f"Processing OOS Year: {period.oos_year}")
    year_folder = os.path.join(RESULTS_DIR, str(period.oos_year))
    os.makedirs(year_folder, exist_ok=True)

    train = full_data[(full_data['year'] >= period.training_start) & (full_data['year'] <= period.training_end)]
    oos = full_data[full_data['year'] == period.oos_year]

    X_train, y_train = train[FEATURES].fillna(0), train[TARGET]
    X_oos, y_oos = oos[FEATURES].fillna(0), oos[TARGET]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_oos = scaler.transform(X_oos)

    model = SGDRegressor(loss='huber', penalty=None, max_iter=50)
    model.fit(X_train, y_train)

    y_pred_oos = model.predict(X_oos)
    r2_oos = r2_score(y_oos, y_pred_oos)

    pd.DataFrame({'permno': oos['permno'], 'month': oos['month'], 'y_true': y_oos, 'y_pred': y_pred_oos})\
      .to_csv(os.path.join(year_folder, 'oos_predictions.csv'), index=False)

    pos = np.where(y_pred_oos >= np.quantile(y_pred_oos, 0.7), 1, np.where(y_pred_oos <= np.quantile(y_pred_oos, 0.3), -1, 0))
    port_ret = pos * y_oos.values

    pd.DataFrame({'month': oos['month'], 'portfolio_return': port_ret})\
      .to_csv(os.path.join(year_folder, 'portfolio.csv'), index=False)

    top20_idx = np.argsort(np.abs(model.coef_))[-20:][::-1]
    pd.DataFrame({'feature': np.array(FEATURES)[top20_idx], 'coefficient': model.coef_[top20_idx]})\
      .to_csv(os.path.join(year_folder, 'top_20_predictors.csv'), index=False)

    sharpe = port_ret.mean() / port_ret.std() if port_ret.std() != 0 else np.nan
    pd.DataFrame({'sharpe_ratio': [sharpe]}).to_csv(os.path.join(year_folder, 'sharpe_ratio.csv'), index=False)
    pd.DataFrame({'oos_r2': [r2_oos]}).to_csv(os.path.join(year_folder, 'oos_r2.csv'), index=False)

    del train, oos, X_train, X_oos, y_train, y_oos, scaler, model, pos, port_ret, y_pred_oos
    gc.collect()
    gc.collect()

print("Finished all OOS years.")


Block 3: Full GLM

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# PARAMETERS
DATA_DIR = 'INSERT DATA PATH HERE'
RESULTS_DIR = 'INSERT OUTPUT PATH HERE'
VALIDATION_LENGTH = 12
START_YEAR = 1957
END_YEAR = 2021
PENALTY_GRID = [1e-4, 1e-2, 1e-1]
BATCH_SIZE = 10000  # mini-batch size for fitting

os.makedirs(RESULTS_DIR, exist_ok=True)

# Load Data
def load_data(data_dir):
    all_files = glob.glob(os.path.join(data_dir, 'year_*.parquet'))
    dfs = []
    for filename in all_files:
        year = int(os.path.basename(filename).split('_')[1].split('.')[0])
        df = pd.read_parquet(filename)
        df['year'] = year
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

full_data = load_data(DATA_DIR)
print(f"Loaded data shape: {full_data.shape}")

# Features and Target
TARGET = 'ret_excess'
FEATURES = [col for col in full_data.columns if col not in ['ret_excess', 'year', 'month', 'permno']]
full_data = full_data.dropna(subset=[TARGET])

# Estimation Periods
def create_estimation_periods(start_year, end_year, validation_length):
    periods = []
    for oos_year in range(1987, end_year + 1):
        validation_end = oos_year - 1
        validation_start = validation_end - validation_length + 1
        training_start = start_year
        training_end = validation_start - 1

        periods.append({
            'oos_year': oos_year,
            'training_start': training_start,
            'training_end': training_end,
            'validation_start': validation_start,
            'validation_end': validation_end
        })
    return pd.DataFrame(periods)

estimation_periods = create_estimation_periods(START_YEAR, END_YEAR, VALIDATION_LENGTH)

# Prepare storage
oos_r2_scores = []

for idx, period in estimation_periods.iterrows():
    print(f"Processing OOS Year: {period.oos_year}")

    train = full_data[(full_data['year'] >= period.training_start) & (full_data['year'] <= period.training_end)]
    val = full_data[(full_data['year'] >= period.validation_start) & (full_data['year'] <= period.validation_end)]
    oos = full_data[full_data['year'] == period.oos_year]

    X_train, y_train = train[FEATURES], train[TARGET]
    X_val, y_val = val[FEATURES], val[TARGET]
    X_oos, y_oos = oos[FEATURES], oos[TARGET]

    # Impute missing
    X_train = X_train.fillna(X_train.mean())
    X_val = X_val.fillna(X_train.mean())
    X_oos = X_oos.fillna(X_train.mean())

    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_oos_scaled = scaler.transform(X_oos)

    best_val_mse = np.inf
    best_model = None
    best_penalty = None

    for penalty in PENALTY_GRID:
        model = SGDRegressor(loss='huber', penalty='l1', alpha=penalty, learning_rate='invscaling', eta0=0.01, max_iter=1, warm_start=True)

        # Mini-batch training
        n_samples = X_train_scaled.shape[0]
        for epoch in range(3):  # 3 epochs through data
            idxs = np.random.permutation(n_samples)
            for batch_start in range(0, n_samples, BATCH_SIZE):
                batch_idx = idxs[batch_start:batch_start+BATCH_SIZE]
                model.partial_fit(X_train_scaled[batch_idx], y_train.values[batch_idx])

        val_preds = model.predict(X_val_scaled)
        val_mse = mean_squared_error(y_val, val_preds)

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_model = model
            best_penalty = penalty

    # Predict OOS
    y_pred_oos = best_model.predict(X_oos_scaled)
    r2_oos = r2_score(y_oos, y_pred_oos)
    oos_r2_scores.append({'oos_year': period.oos_year, 'oos_r2': r2_oos})

    # Save year results immediately
    year_results = pd.DataFrame({
        'permno': oos['permno'],
        'month': oos['month'],
        'y_true': y_oos,
        'y_pred': y_pred_oos
    })
    year_results.to_csv(os.path.join(RESULTS_DIR, f'oos_predictions_{period.oos_year}.csv'), index=False)

    # Save portfolio
    oos_copy = oos.copy()
    oos_copy['y_pred'] = y_pred_oos
    threshold_long = oos_copy['y_pred'].quantile(0.7)
    threshold_short = oos_copy['y_pred'].quantile(0.3)
    oos_copy['position'] = np.where(oos_copy['y_pred'] >= threshold_long, 1,
                            np.where(oos_copy['y_pred'] <= threshold_short, -1, 0))
    oos_copy['portfolio_return'] = oos_copy['position'] * oos_copy['ret_excess']
    oos_copy[['month', 'portfolio_return']].to_csv(os.path.join(RESULTS_DIR, f'portfolio_{period.oos_year}.csv'), index=False)

    # Garbage collection
    del train, val, oos, X_train, X_val, X_oos, y_train, y_val, y_oos, scaler, model, best_model
    gc.collect()

# Save R2 scores
r2_df = pd.DataFrame(oos_r2_scores)
r2_df.to_csv(os.path.join(RESULTS_DIR, 'oos_r2_scores.csv'), index=False)
print("Finished all OOS years.")
